|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>Detokenization<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: the streaming detokenizer<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B')
print('vocab', tokenizer.vocab_size)

Write the streaming detokenizer.

No GPU, no tensors, and no interesting arithmetic. This stage is boring and it
is the source of most user-visible bugs in real servers, which is a fair
trade for one afternoon.

# Exercise 1: emit the difference, not the token

A character can be split across two tokens, so decoding one token at a time
produces replacement characters. Decode the prefix instead and emit whatever
is new.

In [ ]:
class IncrementalDetokenizer:
  """Turn a stream of token ids into a stream of complete characters."""
  def __init__(self, tokenizer):
    self.tokenizer = tokenizer
    self.token_ids = []
    self.num_emitted = 0      # the characters that the client has already

  def push(self, token_id):
    """-> the NEW text that this token completes. It can be ''."""
    self.token_ids.append(token_id)
    text = self.tokenizer.decode(self.token_ids)
    # A replacement character at the end means that the last character is
    # not complete. Hold it back. The next token completes it.
    if text.endswith('\ufffd'):
      return ''
    return self._emit_from(text)

  def flush(self):
    """The stream ended. Send the rest, complete or not."""
    return self._emit_from(self.tokenizer.decode(self.token_ids))

  def _emit_from(self, text):
    new_text = text[self.num_emitted:]
    self.num_emitted = len(text)
    return new_text

def stream_text(tokenizer, token_ids):
  """Send token_ids through a new detokenizer. -> all the text that it sent."""
  detokenizer = IncrementalDetokenizer(tokenizer)
  pushed = ''.join(detokenizer.push(token_id) for token_id in token_ids)
  return pushed + detokenizer.flush()

token_ids = tokenizer('hello world', add_special_tokens=False).input_ids
detokenizer = IncrementalDetokenizer(tokenizer)
print([detokenizer.push(token_id) for token_id in token_ids])

# Exercise 2: prove it with a fuzz test

The contract is an equality. Test it that way.

In [ ]:
import random
# The contract is an equality: the streamed text == the batch-decoded text.
# Prove it on many random token sequences, not on examples.
rng = random.Random(0)
failures = 0
for _ in range(500):
  token_ids = [rng.randrange(tokenizer.vocab_size) for _ in range(rng.randint(1, 40))]
  if stream_text(tokenizer, token_ids) != tokenizer.decode(token_ids):
    failures += 1
print(f'{failures} failures in 500 random sequences')

hard = ['\U0001F468\u200d\U0001F469\u200d\U0001F467\u200d\U0001F466',
        '\U0001F3F3\ufe0f\u200d\U0001F308', 'caf\u00e9', '\u65e5\u672c\u8a9e',
        '\uc548\ub155\ud558\uc138\uc694']
for text in hard:
  token_ids = tokenizer(text, add_special_tokens=False).input_ids
  matches = stream_text(tokenizer, token_ids) == tokenizer.decode(token_ids)
  print(f'{matches}  {text!r}')

# Exercise 3: stop strings that straddle

The model emits ` EN` then `D`. Neither token contains `END`.

In [ ]:
def held_length(held, stop):
  """The length of the longest end of `held` that is a start of `stop`.
  That text can still become the stop string, so do not emit it yet."""
  for length in range(min(len(stop) - 1, len(held)), 0, -1):
    if held.endswith(stop[:length]):
      return length
  return 0

def stream_until_stop(tokenizer, token_ids, stop):
  """Emit text until `stop` appears. Never emit a start of `stop`.
  -> (the emitted text, True if the stop string appeared).

  The stop string can cross a token boundary, so you cannot look at tokens.
  And you cannot emit early: 'EN' must not show before you know that it
  was the start of 'END'.
  """
  detokenizer = IncrementalDetokenizer(tokenizer)
  emitted, held = [], ''
  for token_id in token_ids:
    held += detokenizer.push(token_id)
    stop_at = held.find(stop)
    if stop_at >= 0:
      emitted.append(held[:stop_at])
      return ''.join(emitted), True
    keep = held_length(held, stop)
    emitted.append(held[:len(held) - keep])
    held = held[len(held) - keep:]
  return ''.join(emitted) + held + detokenizer.flush(), False

cases = [('Answer: yes. END OF LINE', 'END'),
         ('nothing to stop for', 'END'),
         ('the ENDING is near', 'END')]
for text, stop in cases:
  token_ids = tokenizer(text, add_special_tokens=False).input_ids
  emitted, stopped = stream_until_stop(tokenizer, token_ids, stop)
  print(f'{text!r}\n  -> {emitted!r}  stopped={stopped}')

### The third case is the one that matters

`'the ENDING is near'` contains `END`, so the stream stops after `the `. That
is correct, and it still surprises somebody. A stop string is a substring
match, and not a word match. A user who sets `stop=['\n']` is sometimes
astonished by what counts as a newline.

Your implementation has three properties. None of them is optional:

- **It never emits a prefix of the stop string.** If it does, `EN` appears in
  the browser of the user for one frame, and then the stream ends.
- **It decodes prefixes, and not tokens.** So it never shows a replacement
  character for one character that two tokens share.
- **An equality covers it, and not a set of examples.** The streamed output
  must equal the batch-decoded output, for all token sequences.

This stage has no interesting arithmetic and no kernel. Do it carefully. These
are the bugs that users see. A fuzz test on a pure function is also the
cheapest correctness that you can buy.

    ./vc guide 14